# Topic-Informed Intermediate Reranking (RQ3)

Three-stage pipeline:
1. Bi-encoder baseline (precomputed top-100 FAISS results)
2. Topic-informed intermediate reranking via linear fusion
3. Cross-encoder reranking

## 1. Setup

In [1]:
!pip install -q "transformers==4.46.3" "sentence-transformers==3.3.1" faiss-cpu


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import json
import pickle
import gc
import os
import itertools
from pathlib import Path
from collections import defaultdict
import faiss

import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")

PyTorch : 2.4.1+cu124
Device  : cuda


## 2. Configuration

In [ ]:
DATA_DIR   = Path('workspace/embeddings')
TOPIC_DIR  = Path('')       
PSC_TOPIC_DIR = TOPIC_DIR / 'psc_fastopic'
OUTPUT_DIR = Path('./retrieval_rq3_topic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Retrieval inputs ──────────────────────────────────────────────────────────
BDC_CHUNKS_JSON      = DATA_DIR / 'VD_bdc_chunks.json'
PSC_CHUNKS_JSON      = DATA_DIR / 'VD_psc_chunks.json'
PSC_CHAPTERS_JSON    = 'psc_chunks_with_chapters.json'  # chunk_id → chapter_id
VALIDATION_SET_JSON  = DATA_DIR / 'ir_ground_truth.json'
ALL_BDC_RETRIEVAL_PKL = DATA_DIR / 'all_bdc_retrieval_e5_top100.pkl'

# ── Embeddings (reused from RQ1) ──────────────────────────────────────────────
PSC_EMBEDDINGS_NPY   = DATA_DIR / 'psc_embeddings_e5-intertext-chapters.npy'
PSC_IDS_JSON         = DATA_DIR / 'psc_ids_e5-intertext-chapters.json'
BDC_EMBEDDINGS_NPY   = DATA_DIR / 'bdc_embeddings_e5.npy'
BDC_IDS_JSON         = DATA_DIR / 'bdc_ids_e5.json'

# ── Topic model outputs ───────────────────────────────────────────────────────
BDC_THETA_NPY        = TOPIC_DIR / 'theta_D_seeded_regest_K20_seed43.npy'
BDC_THETA_IDS_JSON   = TOPIC_DIR / 'theta_D_letter_ids.json'   # produced by mapping script
BDC_TOPIC_WORDS_JSON = TOPIC_DIR / 'topics_D_seeded_regest_K20_seed43.json'
PSC_THETA_NPY        = TOPIC_DIR / 'k020_theta.npy'
PSC_THETA_IDS_JSON   = TOPIC_DIR / 'k020_chapter_ids.json'
PSC_TOPIC_WORDS_JSON = TOPIC_DIR / 'k020_topics.json'

# ── Models ────────────────────────────────────────────────────────────────────
ENCODER_MODEL      = 'julian-schelb/multilingual-e5-large-emb-lat-intertext-v1'
CROSS_ENCODER_MODEL = 'julian-schelb/PhilBerta-class-latin-intertext-v1'

# ── Noise topics to exclude from BDC theta before projection ─────────────────
# Topics 9 and 10 in run D are degenerate tokenisation artefacts (confirmed via UMAP)
BDC_NOISE_TOPICS = {9, 10}

# ── Hyperparameters ───────────────────────────────────────────────────────────
RETRIEVAL_TOP_K  = 100    # bi-encoder candidates passed to cross-encoder
FINAL_K          = 20     # cross-encoder output
RERANK_BATCH     = 256
LAMBDA_VALUES    = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]   # sweep; 0.0 = baseline

token = os.getenv('HF_TOKEN')
print('Configuration loaded.')

Configuration loaded.


## 3. Load data

In [4]:
def load_chunks(path):
    with open(path) as f:
        data = json.load(f)
    chunks = data['chunks'] if isinstance(data, dict) and 'chunks' in data else data
    return {c['chunk_id']: c['text'] for c in chunks}

print('Loading chunk texts...')
bdc_text_lookup = load_chunks(BDC_CHUNKS_JSON)
psc_text_lookup = load_chunks(PSC_CHUNKS_JSON)
print(f'  BDC chunks : {len(bdc_text_lookup):,}')
print(f'  PSC chunks : {len(psc_text_lookup):,}')

Loading chunk texts...
  BDC chunks : 19,466
  PSC chunks : 46,408


In [5]:
# PSC chunk → chapter mapping
print('Loading PSC chunk→chapter mapping...')
with open(PSC_CHAPTERS_JSON) as f:
    psc_chapters_raw = json.load(f)

# Normalise to {chunk_id: chapter_id} regardless of source format
# Adjust key names if your JSON uses different fields
if isinstance(psc_chapters_raw, list):
    psc_chunk_to_chapter = {
        c['chunk_id']: c['chapter_id']
        for c in psc_chapters_raw
        if 'chunk_id' in c and 'chapter_id' in c
    }
elif isinstance(psc_chapters_raw, dict):
    # Could be {chunk_id: chapter_id} directly or {chunks: [...]}
    if 'chunks' in psc_chapters_raw:
        psc_chunk_to_chapter = {
            c['chunk_id']: c['chapter_id']
            for c in psc_chapters_raw['chunks']
        }
    else:
        psc_chunk_to_chapter = psc_chapters_raw

print(f'  PSC chunk→chapter entries: {len(psc_chunk_to_chapter):,}')
sample_key = next(iter(psc_chunk_to_chapter))
print(f'  Sample: {sample_key} → {psc_chunk_to_chapter[sample_key]}')

Loading PSC chunk→chapter mapping...
  PSC chunk→chapter entries: 209,693
  Sample: tlg1443.tlg001.1st1K-grc1_window_0 → tlg1443.tlg001.1st1K-grc1_chap_0


In [6]:
print('Loading validation set...')
with open(VALIDATION_SET_JSON) as f:
    validation_set = json.load(f)
ground_truth = validation_set['ground_truth']
query_ids    = [e['query_chunk_id'] for e in ground_truth]
print(f'  Queries : {len(query_ids)}')
print(f'  Explicit: {sum(1 for e in ground_truth if e["reference_type"] == "explicit")}')
print(f'  Implicit: {sum(1 for e in ground_truth if e["reference_type"] == "implicit")}')

Loading validation set...
  Queries : 100
  Explicit: 50
  Implicit: 50


In [7]:
print('Loading precomputed FAISS baseline...')
with open(ALL_BDC_RETRIEVAL_PKL, 'rb') as f:
    all_bdc_retrieval = pickle.load(f)

retrieval_results = {
    qid: all_bdc_retrieval[qid]
    for qid in query_ids
    if qid in all_bdc_retrieval
}
missing = set(query_ids) - set(retrieval_results)
if missing:
    print(f'WARNING: {len(missing)} queries missing from baseline')
else:
    print(f'  All {len(retrieval_results)} validation queries loaded.')

sample = query_ids[0]
print(f'  Top-3 for {sample}:')
for cid, score in retrieval_results[sample][:3]:
    print(f'    {score:.4f}  {cid}')

Loading precomputed FAISS baseline...
  All 100 validation queries loaded.
  Top-3 for 10015_sent_188_190:
    0.8364  022_Hieronymus-Stridonensis_Epistolae_window_755
    0.8268  035_Augustinus-Hipponensis_In-Joannis-evangelium-tractatus-CXXIV_window_2094
    0.8254  024_Hieronymus-Stridonensis_Commentaria-in-Isaiam_window_1794


## 4. Load topic model outputs

In [8]:
print('Loading BDC theta (run D, K=20)...')
bdc_theta = np.load(BDC_THETA_NPY).astype(np.float32)      # (11837, 20)
with open(BDC_THETA_IDS_JSON) as f:
    bdc_theta_letter_ids = json.load(f)                      # list of letter_ids, len 11837
bdc_letter_to_theta_row = {lid: i for i, lid in enumerate(bdc_theta_letter_ids)}
print(f'  BDC theta shape : {bdc_theta.shape}')
print(f'  BDC theta IDs   : {len(bdc_theta_letter_ids)}')

print('Loading PSC theta (K=20)...')
psc_theta = np.load(PSC_THETA_NPY).astype(np.float32)       # (24075, 20)
with open(PSC_THETA_IDS_JSON) as f:
    psc_theta_chapter_ids = json.load(f)                     # list of chapter_ids, len 24075
psc_chapter_to_theta_row = {cid: i for i, cid in enumerate(psc_theta_chapter_ids)}
print(f'  PSC theta shape : {psc_theta.shape}')
print(f'  PSC theta IDs   : {len(psc_theta_chapter_ids)}')

print('Loading topic word lists...')
with open(BDC_TOPIC_WORDS_JSON) as f:
    bdc_topic_words = json.load(f)   # list of 20 space-separated strings
with open(PSC_TOPIC_WORDS_JSON) as f:
    psc_topic_words = json.load(f)
print(f'  BDC topics: {len(bdc_topic_words)}  PSC topics: {len(psc_topic_words)}')

Loading BDC theta (run D, K=20)...
  BDC theta shape : (11837, 20)
  BDC theta IDs   : 11837
Loading PSC theta (K=20)...
  PSC theta shape : (24075, 20)
  PSC theta IDs   : 24075
Loading topic word lists...
  BDC topics: 20  PSC topics: 20


## 5. Build cross-corpus coupling matrix A

`A[k, j] = cos(centroid_BDC_k, centroid_PSC_j)`

Centroids reconstructed by encoding top-10 topic words via the shared encoder

In [9]:
print(f'Loading encoder: {ENCODER_MODEL}')
encoder = SentenceTransformer(ENCODER_MODEL, device=DEVICE, token=token)
print('Encoder loaded.')

Loading encoder: julian-schelb/multilingual-e5-large-emb-lat-intertext-v1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/318 [00:00<?, ?B/s]

You try to use a model that was created with version 5.2.0, however, your version is 3.3.1. This might cause unexpected behavior or errors. In that case, try to update to the latest version.





README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/544 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Encoder loaded.


In [10]:
def encode_topic_centroids(topic_words, encoder, top_n=10):
    """
    Encode top-N words per topic as a single passage string.
    Returns normalised centroid matrix (K, D).
    """
    strings = []
    for entry in topic_words:
        words = entry.split()[:top_n] if isinstance(entry, str) else entry[:top_n]
        strings.append('passage: ' + ' '.join(words))

    centroids = encoder.encode(
        strings,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return centroids.astype(np.float32)   # (K, D)


print('Encoding BDC topic centroids...')
bdc_centroids = encode_topic_centroids(bdc_topic_words, encoder)  # (20, 1024)
print(f'  BDC centroids: {bdc_centroids.shape}')

print('Encoding PSC topic centroids...')
psc_centroids = encode_topic_centroids(psc_topic_words, encoder)  # (20, 1024)
print(f'  PSC centroids: {psc_centroids.shape}')

Encoding BDC topic centroids...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  BDC centroids: (20, 1024)
Encoding PSC topic centroids...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  PSC centroids: (20, 1024)


In [11]:
# Coupling matrix A: (K_B, K_P) cosine similarities
# Embeddings are already L2-normalised, so dot product = cosine similarity
A = bdc_centroids @ psc_centroids.T   # (20, 20)
print(f'Coupling matrix A shape: {A.shape}')
print(f'  Min: {A.min():.4f}  Max: {A.max():.4f}  Mean: {A.mean():.4f}')

# Zero out noise topic rows before projection
# This prevents degenerate BDC topics from corrupting θ̃ᴾ(l)
noise_mask = np.ones(A.shape[0], dtype=np.float32)
for t in BDC_NOISE_TOPICS:
    noise_mask[t] = 0.0
    print(f'  Zeroed BDC noise topic {t}: {bdc_topic_words[t][:60]}')
A_masked = A * noise_mask[:, None]   # broadcast: zero out noise rows
print(f'Noise topics zeroed: {BDC_NOISE_TOPICS}')

Coupling matrix A shape: (20, 20)
  Min: 0.6222  Max: 0.8900  Mean: 0.7557
  Zeroed BDC noise topic 9: significium seinis acia landes sentia molus satio copus papo
  Zeroed BDC noise topic 10: admirallius milliarus praesidiarius sonitus reginus prelium 
Noise topics zeroed: {9, 10}


## 6. Helper functions

In [12]:
def letter_id_from_chunk_id(chunk_id):
    """'10015_sent_0' → 'file10015'"""
    return 'file' + chunk_id.split('_')[0]


def get_bdc_theta(letter_id):
    """
    Return the topic distribution θᴮ(l) for a letter.
    Returns None if the letter was not in the FASTopic corpus (empty doc).
    """
    row = bdc_letter_to_theta_row.get(letter_id)
    if row is None:
        return None
    return bdc_theta[row]   # (20,)


def project_to_psc(theta_b, A_masked):
    """
    Project BDC letter distribution into PSC topic space.
    θ̃ᴾ(l) = θᴮ(l) · A_masked
    Renormalises after noise topic removal so projection sums to ~1.
    """
    # Zero out noise topics in the letter distribution too
    theta_clean = theta_b.copy()
    for t in BDC_NOISE_TOPICS:
        theta_clean[t] = 0.0
    s = theta_clean.sum()
    if s > 0:
        theta_clean /= s   # renormalise
    projected = theta_clean @ A_masked   # (20,)
    norm = np.linalg.norm(projected)
    if norm > 0:
        projected /= norm
    return projected


def get_psc_chapter_theta(chapter_id):
    """
    Return the topic distribution θᴾ(chapter) for a PSC chapter.
    Returns None if chapter not in theta index.
    """
    row = psc_chapter_to_theta_row.get(chapter_id)
    if row is None:
        return None
    v = psc_theta[row].copy()
    norm = np.linalg.norm(v)
    if norm > 0:
        v /= norm
    return v


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(results, validation_set, k_values=[5, 10, 20, 50, 100]):
    entries = validation_set['ground_truth']

    def _compute(subset):
        metrics = {k: {'recall': [], 'mrr': []} for k in k_values}
        for entry in subset:
            qid    = entry['query_chunk_id']
            gt_ids = set(entry['relevant_chunks'])
            if qid not in results:
                continue
            retrieved = [cid for cid, _ in results[qid]]
            for k in k_values:
                top_k = retrieved[:k]
                metrics[k]['recall'].append(1.0 if any(c in gt_ids for c in top_k) else 0.0)
                mrr = 0.0
                for rank, cid in enumerate(top_k, 1):
                    if cid in gt_ids:
                        mrr = 1.0 / rank
                        break
                metrics[k]['mrr'].append(mrr)
        summary = {}
        for k in k_values:
            r = metrics[k]['recall']
            m = metrics[k]['mrr']
            summary[f'Recall@{k}'] = round(sum(r)/len(r), 4) if r else None
            summary[f'MRR@{k}']    = round(sum(m)/len(m), 4) if m else None
        return summary

    strata = {
        'all'     : entries,
        'explicit': [e for e in entries if e['reference_type'] == 'explicit'],
        'implicit': [e for e in entries if e['reference_type'] == 'implicit'],
    }
    return {s: _compute(sub) for s, sub in strata.items()}


def print_metrics(label, strat):
    print(f'\n── {label} ──')
    print(f'  {"Metric":<12} {"all":>8} {"explicit":>10} {"implicit":>10}')
    for metric in ['Recall@20', 'Recall@100', 'MRR@20']:
        a = strat['all'].get(metric, '-')
        e = strat['explicit'].get(metric, '-')
        i = strat['implicit'].get(metric, '-')
        print(f'  {metric:<12} {str(a):>8} {str(e):>10} {str(i):>10}')


print('Helper functions defined.')

Helper functions defined.


## 7. Precompute projections for all validation queries

For each validation query chunk:
- Resolve its parent letter
- Look up `θᴮ(l)`, zero out noise topics, renormalise
- Project: `θ̃ᴾ(l) = θᴮ(l) · A_masked`
- Cache the result

In [13]:
print('Precomputing letter projections for validation queries...')
query_projected = {}   # query_chunk_id → θ̃ᴾ(l) or None
missing_theta   = []

for qid in query_ids:
    lid     = letter_id_from_chunk_id(qid)
    theta_b = get_bdc_theta(lid)
    if theta_b is None:
        query_projected[qid] = None
        missing_theta.append(qid)
    else:
        query_projected[qid] = project_to_psc(theta_b, A_masked)

print(f'  Projected : {len(query_ids) - len(missing_theta)}')
print(f'  Missing   : {len(missing_theta)} (letter not in FASTopic corpus — will use baseline only)')
if missing_theta:
    print(f'  Examples  : {missing_theta[:3]}')

Precomputing letter projections for validation queries...
  Projected : 100
  Missing   : 0 (letter not in FASTopic corpus — will use baseline only)


## 8. Topic-informed reranking

In [ ]:
# ── Topic-FAISS + RRF  ─────────────────────────────────────────────────
# Build a 20-dim FAISS index over PSC chapter theta distributions
# Query with θ̃ᴾ(l) → top-K chapters → expand 
# RRF-fuse with baseline bi-encoder chunk list
# This allows R@100 to move as topically retrieved chunks enter the pool

import faiss
from collections import defaultdict

# ── Hyperparameters ───────────────────────────────────────────────────────────
TOP_K_CHAPTERS  = 50    # chapters retrieved by topic FAISS per query
TOP_K_CHUNKS_PER_CHAPTER = 5   # chunks taken per retrieved chapter
RRF_K           = 60    # RRF standard constant

# ── 1. Build chapter → chunks inverted index ──────────────────────────────────
print('Building chapter → chunks inverted index...')
chapter_to_chunks = defaultdict(list)
for c in data['chunks']:
    chapter_to_chunks[c['chapter_id']].append(c['chunk_id'])

# Sanity check: exclude the pathological giant chapter
giant = max(chapter_to_chunks, key=lambda k: len(chapter_to_chunks[k]))
print(f'  Largest chapter: {giant} ({len(chapter_to_chunks[giant])} chunks) — will be excluded')
EXCLUDE_CHAPTERS = {giant}

print(f'  Total chapters with chunks: {len(chapter_to_chunks)}')

# ── 2. Build PSC chapter-level FAISS index (20-dim) ──────────────────────────
print('Building 20-dim PSC chapter theta FAISS index...')

# Only index chapters that (a) appear in chunk file and (b) are not excluded
valid_chapter_ids = [
    cid for cid in psc_theta_chapter_ids
    if cid in chapter_to_chunks and cid not in EXCLUDE_CHAPTERS
]
valid_chapter_rows = [psc_chapter_to_theta_row[cid] for cid in valid_chapter_ids]

# Extract and L2-normalise for cosine similarity
chapter_vecs = psc_theta[valid_chapter_rows].astype(np.float32)
norms = np.linalg.norm(chapter_vecs, axis=1, keepdims=True)
chapter_vecs_norm = chapter_vecs / np.maximum(norms, 1e-12)

dim = chapter_vecs_norm.shape[1]   # 20
topic_index = faiss.IndexFlatIP(dim)   # inner product = cosine on normalised vecs
topic_index.add(chapter_vecs_norm)
print(f'  Topic FAISS index: {topic_index.ntotal} chapters × {dim} dims')

# ── 3. RRF helper ─────────────────────────────────────────────────────────────
def rrf_fuse(list_a, list_b, k=RRF_K):
    """
    list_a: [(id, score), ...] ranked by score desc — baseline bi-encoder chunks
    list_b: [(id, score), ...] ranked by score desc — topic-retrieved chunks
    Returns fused list [(id, rrf_score), ...] sorted desc.
    """
    scores = defaultdict(float)
    for rank, (cid, _) in enumerate(list_a, 1):
        scores[cid] += 1.0 / (k + rank)
    for rank, (cid, _) in enumerate(list_b, 1):
        scores[cid] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# ── 4. Topic-FAISS retrieval per query ───────────────────────────────────────
print('Running topic-FAISS + RRF fusion...')

def topic_faiss_rrf(retrieval_results, top_k_chapters=TOP_K_CHAPTERS,
                    chunks_per_chapter=TOP_K_CHUNKS_PER_CHAPTER):
    fused_results = {}
    no_projection = 0
    new_chunks_total = 0

    for qid, baseline in tqdm(retrieval_results.items(), desc='Topic-FAISS RRF'):
        theta_proj = query_projected.get(qid)   # θ̃ᴾ(l), shape (20,) or None

        if theta_proj is None:
            fused_results[qid] = baseline[:RETRIEVAL_TOP_K]
            no_projection += 1
            continue

        # Normalise query vector
        q_vec = theta_proj.copy().astype(np.float32)
        norm  = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec /= norm
        q_vec = q_vec.reshape(1, -1)

        # FAISS search over chapter theta index
        sims, idxs = topic_index.search(q_vec, top_k_chapters)

        # Expand chapters → chunk ranked list
        topic_chunks = []
        seen = set()
        for sim, idx in zip(sims[0], idxs[0]):
            if idx < 0:
                continue
            chapter_id = valid_chapter_ids[idx]
            chunks = chapter_to_chunks[chapter_id]
            for cid in chunks[:chunks_per_chapter]:
                if cid not in seen:
                    topic_chunks.append((cid, float(sim)))
                    seen.add(cid)

        # Count genuinely new chunks not in baseline top-100
        baseline_ids = {cid for cid, _ in baseline[:RETRIEVAL_TOP_K]}
        new = sum(1 for cid, _ in topic_chunks if cid not in baseline_ids)
        new_chunks_total += new

        # RRF fusion
        fused = rrf_fuse(baseline[:RETRIEVAL_TOP_K], topic_chunks)
        fused_results[qid] = fused

    print(f'  Queries with no projection  : {no_projection}')
    print(f'  Total new chunks introduced : {new_chunks_total}')
    print(f'  Mean new chunks per query   : {new_chunks_total/len(retrieval_results):.1f}')
    return fused_results


# ── 5. Sweep TOP_K_CHAPTERS ──────────────────────────────────────────────────
print('\n=== Topic-FAISS + RRF sweep ===')
topic_rrf_rows = []

for top_k_ch in [10, 25, 50, 100]:
    fused = topic_faiss_rrf(retrieval_results, top_k_chapters=top_k_ch)
    strat = evaluate(fused, validation_set)
    row = {
        'top_k_chapters': top_k_ch,
        'R@20_all'      : strat['all']['Recall@20'],
        'R@100_all'     : strat['all']['Recall@100'],
        'MRR@20_all'    : strat['all']['MRR@20'],
        'R@20_expl'     : strat['explicit']['Recall@20'],
        'R@100_expl'    : strat['explicit']['Recall@100'],
        'R@20_impl'     : strat['implicit']['Recall@20'],
        'R@100_impl'    : strat['implicit']['Recall@100'],
    }
    topic_rrf_rows.append((top_k_ch, fused, strat, row))
    print_metrics(f'topic-FAISS RRF top_k_ch={top_k_ch}', strat)

best_topic_rrf = max(topic_rrf_rows, key=lambda x: x[3]['R@100_all'])
print(f'\nBest top_k_chapters by R@100_all: {best_topic_rrf[0]}')
print(f'  R@100={best_topic_rrf[3]["R@100_all"]}  MRR@20={best_topic_rrf[3]["MRR@20_all"]}')

Building chapter → chunks inverted index...
  Largest chapter: None (7873 chunks) — will be excluded
  Total chapters with chunks: 20802
Building 20-dim PSC chapter theta FAISS index...
  Topic FAISS index: 19697 chapters × 20 dims
Running topic-FAISS + RRF fusion...

=== Topic-FAISS + RRF sweep ===


Topic-FAISS RRF:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection  : 0
  Total new chunks introduced : 4425
  Mean new chunks per query   : 44.2

── topic-FAISS RRF top_k_ch=10 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.44       0.78        0.1
  MRR@20         0.2404     0.4705     0.0102


Topic-FAISS RRF:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection  : 0
  Total new chunks introduced : 10775
  Mean new chunks per query   : 107.8

── topic-FAISS RRF top_k_ch=25 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.44       0.78        0.1
  MRR@20         0.2345     0.4589     0.0101


Topic-FAISS RRF:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection  : 0
  Total new chunks introduced : 20803
  Mean new chunks per query   : 208.0

── topic-FAISS RRF top_k_ch=50 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.44       0.78        0.1
  MRR@20         0.2295     0.4489     0.0101


Topic-FAISS RRF:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection  : 0
  Total new chunks introduced : 41767
  Mean new chunks per query   : 417.7

── topic-FAISS RRF top_k_ch=100 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.44       0.78        0.1
  MRR@20         0.2295     0.4489     0.0101

Best top_k_chapters by R@100_all: 10
  R@100=0.44  MRR@20=0.2404


In [18]:
# What are the dominant topics of the ground truth query letters?
from collections import Counter

dominant_topics = []
for qid in query_ids:
    lid = letter_id_from_chunk_id(qid)
    theta_b = get_bdc_theta(lid)
    if theta_b is None:
        continue
    t = theta_b.copy()
    for nt in BDC_NOISE_TOPICS:
        t[nt] = -np.inf
    dominant_topics.append(int(np.argmax(t)))

print(Counter(dominant_topics).most_common())

[(18, 62), (1, 21), (7, 5), (0, 4), (15, 3), (6, 2), (14, 1), (16, 1), (3, 1)]


In [19]:
# What PSC topics do the ground truth relevant chunks actually belong to?
gt_chunk_ids = set(
    cid
    for entry in ground_truth
    for cid in entry['relevant_chunks']
)

dominant_psc_topics = []
for cid in gt_chunk_ids:
    chapter_id = psc_chunk_to_chapter.get(cid)
    if chapter_id is None:
        continue
    row = psc_chapter_to_theta_row.get(chapter_id)
    if row is None:
        continue
    dominant_psc_topics.append(int(np.argmax(psc_theta[row])))

print(Counter(dominant_psc_topics).most_common())
print(f'Total GT chunks with PSC theta: {len(dominant_psc_topics)} / {len(gt_chunk_ids)}')

[(4, 27), (12, 12), (3, 10), (19, 8), (11, 5), (15, 4), (18, 3), (10, 3), (5, 3), (17, 3), (6, 2), (1, 2), (14, 2), (7, 1), (2, 1)]
Total GT chunks with PSC theta: 86 / 90


In [20]:
# ── Topic-enriched query embedding + fresh FAISS retrieval ───────────────────
# Shift the query chunk embedding toward the PSC topic centroid predicted
# by the BDC letter's dominant topic via coupling matrix A.
#
# q_enriched = norm((1-α)·embed(chunk) + α·psc_centroid_j)
#
# where j = argmax over θ̃ᴾ(l) (the projected PSC distribution for the letter)
# Then re-run FAISS with the enriched query vector.
# R@100 can move: different chunks are retrieved entirely.

import faiss as faiss_module


# ── Load BDC chunk embeddings + build chunk→letter lookup ─────────────────────
print('Loading BDC chunk embeddings...')
bdc_embeddings = np.load(BDC_EMBEDDINGS_NPY).astype(np.float32)  # (N_chunks, 1024)
with open(BDC_IDS_JSON) as f:
    bdc_chunk_ids = json.load(f)   # list of chunk_ids aligned with bdc_embeddings rows
bdc_chunk_id_to_row = {cid: i for i, cid in enumerate(bdc_chunk_ids)}
print(f'  BDC embeddings: {bdc_embeddings.shape}')

print('Loading PSC embeddings + building FAISS index...')
psc_embeddings = np.load(PSC_EMBEDDINGS_NPY).astype(np.float32)  # (N_psc_chunks, 1024)
with open(PSC_IDS_JSON) as f:
    psc_chunk_ids_ordered = json.load(f)   # list aligned with psc_embeddings rows
psc_chunk_id_to_row = {cid: i for i, cid in enumerate(psc_chunk_ids_ordered)}

# Build 1024-dim FAISS index over PSC chunk embeddings
psc_dim   = psc_embeddings.shape[1]
psc_index = faiss_module.IndexFlatIP(psc_dim)   # cosine on L2-normalised vecs
# Normalise PSC embeddings before adding
psc_norms = np.linalg.norm(psc_embeddings, axis=1, keepdims=True)
psc_embs_norm = psc_embeddings / np.maximum(psc_norms, 1e-12)
psc_index.add(psc_embs_norm)
print(f'  PSC FAISS index: {psc_index.ntotal} chunks × {psc_dim} dims')

# ── Precompute predicted PSC centroid per validation query ────────────────────
# Use argmax of θ̃ᴾ(l) (projected BDC distribution) to select the dominant
# PSC centroid — this is the thematic prior for the query letter
print('Computing predicted PSC centroids for validation queries...')
query_psc_centroid = {}   # qid → (20,) PSC centroid in 1024-dim space or None

for qid in query_ids:
    theta_proj = query_projected.get(qid)   # θ̃ᴾ(l), shape (20,) already computed
    if theta_proj is None:
        query_psc_centroid[qid] = None
        continue
    # Dominant PSC topic from projected distribution
    dominant_psc = int(np.argmax(theta_proj))
    query_psc_centroid[qid] = psc_centroids[dominant_psc]   # (1024,) L2-normalised

print(f'  Queries with PSC centroid: '
      f'{sum(1 for v in query_psc_centroid.values() if v is not None)}')
dominant_counts = Counter(
    int(np.argmax(query_projected[qid]))
    for qid in query_ids if query_projected.get(qid) is not None
)
print(f'  Dominant PSC topics across queries: {dominant_counts.most_common(5)}')

Loading BDC chunk embeddings...
  BDC embeddings: (961431, 1024)
Loading PSC embeddings + building FAISS index...
  PSC FAISS index: 209693 chunks × 1024 dims
Computing predicted PSC centroids for validation queries...
  Queries with PSC centroid: 100
  Dominant PSC topics across queries: [(4, 100)]


In [ ]:
# Instead of a fixed semantic threshold, weight topic candidate score
# by semantic similarity to the query chunk before RRF fusion:
#   combined_score = topic_sim × sem_sim



TOP_K_CHAPTERS_WEIGHTED = 50
CHUNKS_PER_CHAPTER_WEIGHTED = 30
TOP_M_TOPIC_CHUNKS = [10, 25, 30, 40, 50, 100]   # cap on topic candidates entering RRF

def topic_faiss_rrf_weighted(retrieval_results, top_k_chapters,
                              chunks_per_chapter, top_m):
    """
    Topic-FAISS expansion with semantic-weighted candidate scoring.
    combined_score = topic_sim × sem_sim(query_chunk, psc_chunk)
    Topic candidates are ranked by combined_score and capped at top_m
    before RRF fusion with the baseline.
    """
    fused_results    = {}
    no_projection    = 0
    new_chunks_total = 0
    no_emb_total     = 0

    for qid, baseline in tqdm(
        retrieval_results.items(),
        desc=f'Weighted RRF top_m={top_m}'
    ):
        theta_proj = query_projected.get(qid)

        if theta_proj is None:
            fused_results[qid] = baseline[:RETRIEVAL_TOP_K]
            no_projection += 1
            continue

        # Query chunk embedding
        q_row = bdc_chunk_id_to_row.get(qid)
        if q_row is None:
            fused_results[qid] = baseline[:RETRIEVAL_TOP_K]
            no_projection += 1
            continue
        q_emb = bdc_embeddings[q_row].astype(np.float32)
        q_norm = np.linalg.norm(q_emb)
        if q_norm > 0:
            q_emb /= q_norm

        # Topic-FAISS: retrieve topically similar chapters
        q_vec = theta_proj.copy().astype(np.float32)
        norm  = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec /= norm
        q_vec = q_vec.reshape(1, -1)
        sims, idxs = topic_index.search(q_vec, top_k_chapters)

        # Expand chapters → chunks, score by topic_sim × sem_sim
        topic_chunks_scored = []
        seen = set()

        for topic_sim, idx in zip(sims[0], idxs[0]):
            if idx < 0:
                continue
            chapter_id = valid_chapter_ids[idx]
            for cid in chapter_to_chunks[chapter_id][:chunks_per_chapter]:
                if cid in seen:
                    continue
                seen.add(cid)

                psc_row = psc_chunk_id_to_row.get(cid)
                if psc_row is None:
                    no_emb_total += 1
                    continue

                psc_emb     = psc_embs_norm[psc_row]
                sem_sim     = float(np.dot(q_emb, psc_emb))
                combined    = float(topic_sim) * sem_sim
                topic_chunks_scored.append((cid, combined))

        # Rank by combined score, take top_m
        topic_chunks_scored.sort(key=lambda x: x[1], reverse=True)
        topic_chunks_top = topic_chunks_scored[:top_m]

        # Count genuinely new chunks
        baseline_ids = {cid for cid, _ in baseline[:RETRIEVAL_TOP_K]}
        new_chunks_total += sum(1 for cid, _ in topic_chunks_top
                                if cid not in baseline_ids)

        fused = rrf_fuse(baseline[:RETRIEVAL_TOP_K], topic_chunks_top)
        fused_results[qid] = fused

    print(f'  Queries with no projection       : {no_projection}')
    print(f'  Chunks with no PSC embedding     : {no_emb_total}')
    print(f'  Total new chunks introduced      : {new_chunks_total}')
    print(f'  Mean new chunks per query        : {new_chunks_total/len(retrieval_results):.1f}')
    return fused_results


# ── Sweep ───────────────────────────────────────────────────────────────

N_VALUES = [25, 50, 100, 200]
M_VALUES = [10, 25, 50, 100]
CHUNKS_PER_CHAPTER_JOINT = 20

print('=== Joint n × M sweep: topic-FAISS semantic-weighted RRF ===')
joint_rows = []

for n in N_VALUES:
    for m in M_VALUES:
        fused = topic_faiss_rrf_weighted(
            retrieval_results,
            top_k_chapters=n,
            chunks_per_chapter=CHUNKS_PER_CHAPTER_JOINT,
            top_m=m,
        )
        strat = evaluate(fused, validation_set)
        row = {
            'n'         : n,
            'm'         : m,
            'R@20_all'  : strat['all']['Recall@20'],
            'R@100_all' : strat['all']['Recall@100'],
            'MRR@20_all': strat['all']['MRR@20'],
            'R@20_expl' : strat['explicit']['Recall@20'],
            'R@100_expl': strat['explicit']['Recall@100'],
            'R@20_impl' : strat['implicit']['Recall@20'],
            'R@100_impl': strat['implicit']['Recall@100'],
        }
        joint_rows.append((n, m, fused, strat, row))
        print_metrics(f'n={n} M={m}', strat)

# Best by R@100_all, then MRR@20_all as tiebreaker
best_joint = max(joint_rows, key=lambda x: (x[4]['R@100_impl'], x[4]['MRR@20_all']))
print(f'\nBest config by R@100_impl then MRR@20_all:')
print(f'  n={best_joint[0]}  M={best_joint[1]}')
print(f'  R@100={best_joint[4]["R@100_impl"]}  '
      f'R@20={best_joint[4]["R@20_impl"]}  '
      f'MRR@20={best_joint[4]["MRR@20_all"]}')

# Print summary table
print(f'\n{"n":>6} {"M":>6} {"R@20":>8} {"R@100":>8} {"MRR@20":>8}')
print('─' * 44)
for n, m, _, _, row in joint_rows:
    print(f'{n:>6} {m:>6} '
          f'{row["R@20_all"]:>8} '
          f'{row["R@100_all"]:>8} '
          f'{row["MRR@20_all"]:>8}')

# Save
joint_summary = [row for _, _, _, _, row in joint_rows]
with open(OUTPUT_DIR / 'topic_faiss_weighted_joint_sweep.json', 'w') as f:
    json.dump(joint_summary, f, indent=2)
print(f'\nSaved → {OUTPUT_DIR}/topic_faiss_weighted_joint_sweep.json')

=== Joint n × M sweep: topic-FAISS semantic-weighted RRF ===


Weighted RRF top_m=10:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection       : 0
  Chunks with no PSC embedding     : 0
  Total new chunks introduced      : 973
  Mean new chunks per query        : 9.7

── n=25 M=10 ──
  Metric            all   explicit   implicit
  Recall@20        0.39        0.7       0.08
  Recall@100       0.47       0.78       0.16
  MRR@20         0.2237     0.4359     0.0115


Weighted RRF top_m=25:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection       : 0
  Chunks with no PSC embedding     : 0
  Total new chunks introduced      : 2472
  Mean new chunks per query        : 24.7

── n=25 M=25 ──
  Metric            all   explicit   implicit
  Recall@20        0.39        0.7       0.08
  Recall@100       0.47       0.78       0.16
  MRR@20         0.2236     0.4357     0.0115


...

  Queries with no projection       : 0
  Chunks with no PSC embedding     : 0
  Total new chunks introduced      : 2383
  Mean new chunks per query        : 23.8

── n=200 M=25 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.46       0.78       0.14
  MRR@20         0.1625     0.3147     0.0102


Weighted RRF top_m=50:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection       : 0
  Chunks with no PSC embedding     : 0
  Total new chunks introduced      : 4883
  Mean new chunks per query        : 48.8

── n=200 M=50 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.45       0.78       0.12
  MRR@20         0.1625     0.3147     0.0102


Weighted RRF top_m=100:   0%|          | 0/100 [00:00<?, ?it/s]

  Queries with no projection       : 0
  Chunks with no PSC embedding     : 0
  Total new chunks introduced      : 9883
  Mean new chunks per query        : 98.8

── n=200 M=100 ──
  Metric            all   explicit   implicit
  Recall@20        0.38        0.7       0.06
  Recall@100       0.45       0.78       0.12
  MRR@20         0.1625     0.3147     0.0102

Best config by R@100_impl then MRR@20_all:
  n=25  M=10
  R@100=0.16  R@20=0.08  MRR@20=0.2237

     n      M     R@20    R@100   MRR@20
────────────────────────────────────────────
    25     10     0.39     0.47   0.2237
    25     25     0.39     0.47   0.2236
    25     50     0.39     0.45   0.2236
    25    100     0.39     0.45   0.2236
    50     10     0.38     0.46   0.2164
    50     25     0.38     0.47   0.2163
    50     50     0.38     0.45   0.2163
    50    100     0.38     0.45   0.2163
   100     10     0.38     0.46   0.1721
   100     25     0.38     0.46   0.1719
   100     50     0.38     0.45   0.1719
 

In [68]:
import faiss
from collections import defaultdict

# ── 1. Reload psc_chunks_with_chapters if needed ──────────────────────────────
with open(PSC_CHAPTERS_JSON) as f:
    data = json.load(f)
print(f'PSC chunks loaded: {len(data["chunks"]):,}')

# ── 2. Build chapter → chunks inverted index ──────────────────────────────────
chapter_to_chunks = defaultdict(list)
for c in data['chunks']:
    chapter_to_chunks[c['chapter_id']].append(c['chunk_id'])

giant = max(chapter_to_chunks, key=lambda k: len(chapter_to_chunks[k]))
EXCLUDE_CHAPTERS = {giant}
print(f'Largest chapter: {giant} ({len(chapter_to_chunks[giant])} chunks) — excluded')
print(f'Total chapters with chunks: {len(chapter_to_chunks)}')

# ── 3. Check overlap ──────────────────────────────────────────────────────────
overlap = sum(1 for cid in psc_theta_chapter_ids if cid in chapter_to_chunks)
print(f'Overlap between theta IDs and chunk chapter IDs: {overlap}')
print(f'First 3 theta IDs  : {psc_theta_chapter_ids[:3]}')
print(f'First 3 chapter IDs: {list(chapter_to_chunks.keys())[:3]}')

PSC chunks loaded: 209,693
Largest chapter: None (7873 chunks) — excluded
Total chapters with chunks: 20802
Overlap between theta IDs and chunk chapter IDs: 19697
First 3 theta IDs  : ['tlg1622.tlg001.1st1K-grc1_chap_0', 'tlg1205.tlg001.perseus-grc1_chap_1', 'tlg1205.tlg001.perseus-grc1_chap_2']
First 3 chapter IDs: ['tlg1443.tlg001.1st1K-grc1_chap_0', 'tlg1443.tlg001.1st1K-grc1_chap_1', 'tlg1443.tlg001.1st1K-grc1_chap_2']


In [82]:
print(f'PSC embedding index size : {len(psc_chunk_id_to_row):,}')
print(f'PSC chunks with chapters : {sum(len(v) for v in chapter_to_chunks.values()):,}')

# What fraction of chapter chunks are in the embedding index?
in_index = sum(
    1 for chunks in chapter_to_chunks.values()
    for cid in chunks
    if cid in psc_chunk_id_to_row
)
total = sum(len(v) for v in chapter_to_chunks.values())
print(f'Chapter chunks in embedding index: {in_index:,} / {total:,} ({100*in_index/total:.1f}%)')

# Are the missing chunks systematically from certain chapters?
chapters_fully_missing = sum(
    1 for cid, chunks in chapter_to_chunks.items()
    if not any(c in psc_chunk_id_to_row for c in chunks)
)
print(f'Chapters with zero chunks in embedding index: {chapters_fully_missing:,} / {len(chapter_to_chunks):,}')

PSC embedding index size : 209,693
PSC chunks with chapters : 209,693
Chapter chunks in embedding index: 209,693 / 209,693 (100.0%)
Chapters with zero chunks in embedding index: 0 / 20,802


In [ ]:
# ── Cross-encoder reranking over best topic-weighted RRF (n=25, M=25) ─────────
print('Reranking best topic-weighted RRF (n=25, M=25) with cross-encoder...')

# Retrieve fused results for n=25, M=25 from joint sweep
best_topic_fused_n25_m25 = next(
    fused for n, m, fused, _, _ in joint_rows if n == 25 and m == 25
)

reranked_topic_n25_m25 = rerank_ce(
    best_topic_fused_n25_m25,
    desc='CE topic-weighted RRF n=25 M=25'
)

strat_ce_topic_n25_m25 = evaluate(reranked_topic_n25_m25, validation_set)

print('\n── Cross-encoder results: topic-weighted RRF (n=25, M=25) ──')
print(f'  {"Metric":<14} {"all":>8} {"explicit":>10} {"implicit":>10}')
print('─' * 48)
for metric in ['Recall@20', 'Recall@50', 'Recall@100', 'MRR@20']:
    a = strat_ce_topic_n25_m25['all'].get(metric, '-')
    e = strat_ce_topic_n25_m25['explicit'].get(metric, '-')
    i = strat_ce_topic_n25_m25['implicit'].get(metric, '-')
    print(f'  {metric:<14} {str(a):>8} {str(e):>10} {str(i):>10}')

# Compare against CE baseline
print('\n── Comparison: CE baseline vs CE topic-weighted RRF (n=25, M=25) ──')
print(f'  {"Metric":<14} {"CE base":>10} {"CE topic":>10} {"delta":>8}')
print('─' * 48)
for metric in ['Recall@20', 'Recall@100', 'MRR@20']:
    base  = strat_ce_base['all'].get(metric, 0)
    topic = strat_ce_topic_n25_m25['all'].get(metric, 0)
    delta = round(topic - base, 4) if isinstance(base, float) and isinstance(topic, float) else '-'
    sign  = '+' if isinstance(delta, float) and delta > 0 else ''
    print(f'  {metric:<14} {str(base):>10} {str(topic):>10} {sign+str(delta):>8}')



Reranking best topic-weighted RRF (n=25, M=25) with cross-encoder...


CE topic-weighted RRF n=25 M=25:   0%|          | 0/100 [00:00<?, ?it/s]

In [138]:
# ── Ground truth improvement diagnostic ───────────────────────────────────────
# Find which GT chunks are recovered by topic RRF (n=25, M=25) at R@100
# that were not in the baseline top-100, and show letter text + PSC chunk text
# + BDC dominant topic + PSC chapter dominant topic

print('=== Ground truth improvement analysis: topic RRF vs baseline ===\n')

improvements = []

for entry in ground_truth:
    qid      = entry['query_chunk_id']
    gt_ids   = set(entry['relevant_chunks'])
    ref_type = entry['reference_type']

    baseline_ids  = {cid for cid, _ in retrieval_results.get(qid, [])[:RETRIEVAL_TOP_K]}
    topic_ids     = {cid for cid, _ in best_topic_fused_n25_m25.get(qid, [])[:RETRIEVAL_TOP_K]}

    newly_found = gt_ids & topic_ids - baseline_ids
    lost        = gt_ids & baseline_ids - topic_ids

    if newly_found:
        improvements.append({
            'qid'        : qid,
            'ref_type'   : ref_type,
            'newly_found': newly_found,
            'lost'       : lost,
        })

print(f'Queries with newly found GT chunks : {len(improvements)}')
print(f'Total newly found GT chunks        : {sum(len(x["newly_found"]) for x in improvements)}')
print()

for imp in improvements:
    qid        = imp['qid']
    lid        = letter_id_from_chunk_id(qid)
    ref_type   = imp['ref_type']

    print(f'{"="*70}')
    print(f'Query chunk  : {qid}  ({ref_type})')
    print(f'Letter ID    : {lid}')

    # BDC letter text
    bdc_text = bdc_text_lookup.get(qid, '[not found]')
    print(f'\n── BDC chunk text ──')
    print(bdc_text[:500] + ('...' if len(bdc_text) > 500 else ''))

    # BDC dominant topic
    theta_b = get_bdc_theta(lid)
    if theta_b is not None:
        t = theta_b.copy()
        for nt in BDC_NOISE_TOPICS:
            t[nt] = -np.inf
        dom_bdc = int(np.argmax(t))
        print(f'\n── BDC letter dominant topic ──')
        print(f'  Topic {dom_bdc}: {bdc_topic_words[dom_bdc]}')

    # Newly found GT chunks
    for gt_cid in imp['newly_found']:
        print(f'\n── Newly found GT chunk : {gt_cid} ──')

        # PSC chunk text
        psc_text = psc_text_lookup.get(gt_cid, '[not found]')
        print(f'Text: {psc_text[:500]}' + ('...' if len(psc_text) > 500 else ''))

        # PSC chapter dominant topic
        chapter_id = psc_chunk_to_chapter.get(gt_cid)
        print(f'Chapter ID: {chapter_id}')
        if chapter_id:
            row = psc_chapter_to_theta_row.get(chapter_id)
            if row is not None:
                dom_psc = int(np.argmax(psc_theta[row]))
                print(f'PSC chapter dominant topic {dom_psc}: {psc_topic_words[dom_psc]}')

        # Where did it rank in baseline vs topic RRF?
        baseline_ranked = [cid for cid, _ in retrieval_results.get(qid, [])]
        topic_ranked    = [cid for cid, _ in best_topic_fused_n25_m25.get(qid, [])]

        base_rank  = baseline_ranked.index(gt_cid) + 1 if gt_cid in baseline_ranked else 'not retrieved'
        topic_rank = topic_ranked.index(gt_cid) + 1 if gt_cid in topic_ranked else 'not retrieved'
        print(f'Baseline rank : {base_rank}')
        print(f'Topic RRF rank: {topic_rank}')

    # Lost GT chunks
    if imp['lost']:
        print(f'\n── Lost GT chunks (in baseline but not topic RRF top-100) ──')
        for gt_cid in imp['lost']:
            baseline_ranked = [cid for cid, _ in retrieval_results.get(qid, [])]
            topic_ranked    = [cid for cid, _ in best_topic_fused_n25_m25.get(qid, [])]
            base_rank  = baseline_ranked.index(gt_cid) + 1 if gt_cid in baseline_ranked else 'not retrieved'
            topic_rank = topic_ranked.index(gt_cid) + 1 if gt_cid in topic_ranked else 'not retrieved'
            print(f'  {gt_cid}  baseline rank: {base_rank}  topic RRF rank: {topic_rank}')

    print()

=== Ground truth improvement analysis: topic RRF vs baseline ===

Queries with newly found GT chunks : 1
Total newly found GT chunks        : 1

Query chunk  : 10857_sent_6  (implicit)
Letter ID    : file10857

── BDC chunk text ──
als cum doctore Hieronymo non aquam , sed spiritum lavare animam Deinde diserte sentis pios rem et signum sumere , idque fidei benefitio , impios signum tantum , ideoque signum et signatum non necesse apud sanctos esse juncta , apud quos interim res iustificat , etsi signum absit , modo id non fiat per contemptum , adeoque in sacramentis praecipue spectandam esse rem , quae significatur , et fidem , verbum et institutionem , ex quibus signo omnis accedit authoritas , quod sine illis nullum est .

── BDC letter dominant topic ──
  Topic 0: paulusbriefen authoritate abendmahlsbekenntnis retraktation drucklegung aphorisme saturitas verteidigungsschrift theologenkonvent evangelie

── Newly found GT chunk : 023_Hieronymus-Stridonensis_Dialogus-contra-Luciferianos

In [137]:
# Find the new implicit chunk that topic CE surfaced but baseline CE missed
print('=== New implicit chunk surfaced by topic CE ===\n')

for entry in ground_truth:
    if entry['reference_type'] != 'implicit':
        continue
    qid    = entry['query_chunk_id']
    gt_ids = entry['relevant_chunks']
    lid    = letter_id_from_chunk_id(qid)

    for gt_cid in gt_ids:
        base_rank  = baseline_ranks.get(qid, {}).get(gt_cid, None)
        topic_rank = topic_ranks.get(qid, {}).get(gt_cid, None)

        if base_rank is None and topic_rank is not None:
            # This chunk was NOT in baseline CE top-20 but IS in topic CE top-20
            theta_b = get_bdc_theta(lid)
            bdc_dom, bdc_label = None, 'N/A'
            if theta_b is not None:
                t = theta_b.copy()
                for nt in BDC_NOISE_TOPICS:
                    t[nt] = -np.inf
                bdc_dom   = int(np.argmax(t))
                bdc_label = f'T{bdc_dom}: {topic_label(bdc_topic_words_list[bdc_dom])}'

            chapter_id = psc_chunk_to_chapter.get(gt_cid)
            psc_label  = 'N/A'
            if chapter_id:
                psc_row = psc_chapter_to_theta_row.get(chapter_id)
                if psc_row is not None:
                    psc_dom   = int(np.argmax(psc_theta[psc_row]))
                    psc_label = f'T{psc_dom}: {topic_label(psc_topic_words_list[psc_dom])}'

            print(f'Query chunk  : {qid}')
            print(f'Letter       : {lid}')
            print(f'GT PSC chunk : {gt_cid}')
            print(f'Topic CE rank: {topic_rank}')
            print(f'BDC topic    : {bdc_label}')
            print(f'PSC topic    : {psc_label}')
            print(f'\nLetter text (query chunk):')
            print(f'  {bdc_text_lookup.get(qid, "[not found]")[:600]}')
            print(f'\nPSC source text (GT chunk):')
            print(f'  {psc_text_lookup.get(gt_cid, "[not found]")[:600]}')

        elif base_rank is not None and topic_rank is None:
            # This chunk WAS in baseline CE top-20 but is NOT in topic CE top-20
            print(f'\n[LOST] Query: {qid}  GT: {gt_cid}')
            print(f'  Was at baseline rank {base_rank}, dropped out of topic CE top-20')

=== New implicit chunk surfaced by topic CE ===

Query chunk  : 10857_sent_6
Letter       : file10857
GT PSC chunk : 023_Hieronymus-Stridonensis_Dialogus-contra-Luciferianos_window_23
Topic CE rank: 9
BDC topic    : T0: paulusbriefen, authoritate, abendmahlsbekenntnis, retraktation, drucklegung
PSC topic    : T12: monasterium, gregorius, presbyter, fraternitas, episcopus

Letter text (query chunk):
  als cum doctore Hieronymo non aquam , sed spiritum lavare animam Deinde diserte sentis pios rem et signum sumere , idque fidei benefitio , impios signum tantum , ideoque signum et signatum non necesse apud sanctos esse juncta , apud quos interim res iustificat , etsi signum absit , modo id non fiat per contemptum , adeoque in sacramentis praecipue spectandam esse rem , quae significatur , et fidem , verbum et institutionem , ex quibus signo omnis accedit authoritas , quod sine illis nullum est .

PSC source text (GT chunk):
  Arianos peccata posse dimitti? Quomodo antiquis sordibus anima p

In [139]:
# What does the coupling matrix predict for BDC topic 0?
bdc_topic = 0
coupling_row = A_masked[bdc_topic]
print(f'BDC topic 0 coupling to PSC topics:')
for j, score in enumerate(coupling_row):
    print(f'  PSC topic {j:2d} ({score:.4f}): {psc_topic_words[j][:60]}')

print(f'\nDominant PSC coupling target: topic {int(np.argmax(coupling_row))}')
print(f'PSC topic 12 coupling score : {coupling_row[12]:.4f}')
print(f'Max coupling score          : {coupling_row.max():.4f}')

BDC topic 0 coupling to PSC topics:
  PSC topic  0 (0.7328): august ecclus manicheus expectatio adulescens prolapsio marg
  PSC topic  1 (0.7743): complendus natalitia illumino miserator misericors saluus ex
  PSC topic  2 (0.7177): maii proportio ogdoas aprilis tonus consonantia senarius lun
  PSC topic  3 (0.7309): sadducaei galatas scripturus baptismas barnabas nescitis luc
  PSC topic  4 (0.7808): christus fides spiritus caro gratia apostolus pater iesus de
  PSC topic  5 (0.7646): theodotio ἐλευθερία βούλημα γέννημα saba ἀπιστία ἐπίνοια ier
  PSC topic  6 (0.7329): absurditas eliphaz auersio inuectio phantasia motio stoici p
  PSC topic  7 (0.6918): pecunia mammona auarus sanies rapto messor gleba medicus uer
  PSC topic  8 (0.7190): caesar bellum arma miles imperium antiochus proelium uates m
  PSC topic  9 (0.7191): tentatio culpa patientia delectatio superbia supplicium conc
  PSC topic 10 (0.7338): israhel solomon gedeon sarra exodum iudae chananaeus chanana
  PSC topic 11 (0.